In [2]:
# 파이썬 requests 모듈을 이용한 기상청 데이터 가져오기
import requests, json
from datetime import datetime, timedelta # 날짜와 시간을 다루는 클래스

# 특정 지역 격차 좌표 지정(지역: 시흥시 정왕동)
location = "시흥시 정왕동"
FIXED_NX = 56
FIXED_NY = 122

# 공공데이터 포텔에서 발급받은 서비스 키
SERVICE_KEY = '2d664b77231d79af0e90ebb2bed456dfb5242491138aea9aad065fddc912078f'

# 호출 기준 날짜와 시간
def get_datetime():
    now = datetime.now()
    # 초단기실황은 매시 40분 이후 안정적으로 생성되므로, 40분 기준으로 데이터 요청
    if now.minute <40: # 40분 전이면 '이전 시간 정시' 데이터 요청
        api_time = now - timedelta(hours=1) # timedelta() 날짜나 시간 차이 계산
        base_time = api_time.strftime("%H00") # strftime() 시간을 문자열 포맷으로 변환, 매시간 정시로 표현
        base_date = api_time.strftime("%Y%m%d")
    else: # 40분 이후면 '현재 시간 정시' 데이터 요청
        base_time = now.strftime("%H00")
        base_date = now.strftime("%Y%m%d")
    return base_date, base_time

base_date, base_time = get_datetime()

# 호출할 초단기실황 파라미터
url = 'http://apis.data.go.kr/1360000/VilageFcstInfoService_2.0/getUltraSrtNcst'
params ={'serviceKey' : SERVICE_KEY, 'pageNo' : '1', 'numOfRows' : '1000', 'dataType' : 'JSON', 'base_date' : base_date, 'base_time' : base_time, 'nx' : FIXED_NX, 'ny' : FIXED_NY }

# 기상청으로 응답메시지 받아오기
def forecast():
    # GET 요청, url 주소와 파라미터
    response = requests.get(url,params=params)
    dict_data = response.json() # JSON --> Dict type으로 반환
    #print("response_data:",dict_data)

    weather_data = {}
    # 날씨정보에서 기온, 습도, 강수정보 파싱
    for item in dict_data['response']['body']['items']['item']:
        if item['category'] == 'T1H': # 기온
            weather_data['tmp'] = item['obsrValue']
        if item['category'] == 'REH': # 습도
            weather_data['hum'] = item['obsrValue']
        # PTY(강수형태): 없음(0), 비(1), 빗방울(5)
        if item['category'] == 'PTY':
            weather_data['sky'] = item['obsrValue']
    return weather_data

# 기상청데이터 받아서 가공하기
def proc_weather():
    dict_sky = forecast()
    print(f"날씨데이터: {dict_sky}")
    print(f"\n==== {base_date} {base_time} {location} 실시간 날씨 실황 ====")
    if dict_sky['sky'] != None:
        # 강수형태: 없음(0), 비(1), 비/눈(2), 눈(3), 빗방울(5), 빗방울눈날림(6), 눈날림(7)
        if dict_sky['sky'] == '0':
            print("날씨 : 맑음")
        elif dict_sky['sky'] == '1' or '5':
            print("날씨 : 비 또는 빗방울 떨어짐")
        elif dict_sky['sky'] == '2' or '6':
            print("날씨 : 진눈깨비")
        elif dict_sky['sky'] == '3' or '7':
            print("날씨 : 눈")
    if dict_sky['tmp'] != None:
        print("온도 : ",dict_sky['tmp']+"C")
    if dict_sky['hum'] != None:
        print("습도 : ",dict_sky['hum']+"%")

if __name__ == "__main__":
    proc_weather()

날씨데이터: {'sky': '0', 'hum': '28', 'tmp': '27.2'}

==== 20260518 1400 시흥시 정왕동 실시간 날씨 실황 ====
날씨 : 맑음
온도 :  27.2C
습도 :  28%
